In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

SEED = 509
random_state = np.random.RandomState(SEED)

In [79]:
df = pd.read_csv("final_dataset.csv")

In [80]:
df_camry = df[(df["model"] == "CAMRY") & (df["year"] > 2019)]
df_camry = df_camry.drop(columns=["detail_url", "make", "model", "photo_urls_json", "make_model_year_price_median"])
df_camry.nunique()

vin                          2681
seller_type                     2
auction                         2
sale_date                     384
sold_price                    454
currency                        2
year                            7
mileage                      2515
engine_volume                   2
cylinders                       2
fuel_type                       4
transmission                    3
drive_type                      3
primary_damage                 24
primary_damage_severity         8
secondary_damage               25
secondary_damage_severity       8
dtype: int64

In [111]:
df_camry.describe(include="all")

,sale_date,seller_type,auction,sold_price,year,mileage,engine_volume,fuel_type,primary_damage,primary_damage_severity,secondary_damage,secondary_damage_severity
count,2648,2648,2648,2648.000000,2648.000000,2545.000000,2646.000000,2648,2648,1998.000000,2648,1622.000000
unique,383,2,2,NaN,NaN,NaN,NaN,3,10,NaN,10,NaN
top,01/15/2026 18:00 UTC,Insurance,copart,NaN,NaN,NaN,NaN,GAS,FRONT END,NaN,NOT_RECORDED,NaN
freq,36,2267,1664,NaN,NaN,NaN,NaN,1729,1372,NaN,876,NaN
mean,NaN,NaN,NaN,9372.598943,2022.609517,59258.088016,2.534769,NaN,NaN,3.756507,NaN,3.358508
std,NaN,NaN,NaN,4068.684903,1.887089,51311.350937,0.183230,NaN,NaN,0.489392,NaN,0.719336
min,NaN,NaN,NaN,100.000000,2020.000000,1.000000,2.500000,NaN,NaN,1.000000,NaN,1.000000
25%,NaN,NaN,NaN,6500.000000,2021.000000,23487.000000,2.500000,NaN,NaN,3.500000,NaN,3.000000
50%,NaN,NaN,NaN,8400.000000,2023.000000,51526.000000,2.500000,NaN,NaN,4.000000,NaN,3.500000
75%,NaN,NaN,NaN,11900.000000,2025.000000,81961.000000,2.500000,NaN,NaN,4.000000,NaN,4.000000


In [112]:
fig_price_dist = px.histogram(
    df_camry,
    x="sold_price",
    nbins=40,
    title="Distribution of Camry Sold Prices",
    labels={"sold_price": "Sold Price (USD)", "count": "Number of Vehicles"},
)
fig_price_dist.show()

In [81]:
df_camry = df_camry.drop_duplicates(subset=["vin"])
df_camry.shape

(2681, 17)

In [82]:
df_camry = df_camry.drop(columns=["vin"])

In [83]:
df_camry.insert(0, 'sale_date', df_camry.pop('sale_date'))

In [108]:
cad_to_usd = 0.713508

df_camry["sold_price"] = df_camry["sold_price"].astype(float)

mask = df_camry["currency"] == "CAD"

df_camry.loc[mask, "sold_price"] = df_camry.loc[mask, "sold_price"] * cad_to_usd

df_camry["sold_price"] = df_camry["sold_price"].astype(int)

df_camry.loc[mask, "currency"] = "USD"

df_camry = df_camry.drop(columns=["currency"])

In [85]:
corr_matrix = corr_data[["engine_volume", "cylinders"]].corr()

fig_heatmap = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation: Engine Volume and Cylinders",
)
fig_heatmap.show()

In [86]:
df_camry = df_camry.drop(columns=["cylinders"])

In [87]:
df_camry.head()

,sale_date,seller_type,auction,sold_price,currency,year,mileage,engine_volume,fuel_type,transmission,drive_type,primary_damage,primary_damage_severity,secondary_damage,secondary_damage_severity
335,03/06/2026 17:00 UTC,Insurance,iaai,6600,USD,2021,76491.0,2.5,GAS,AUTOMATIC,FWD,FRONT END,3.5,FRONT END,4.0
515,03/06/2026 18:00 UTC,Insurance,copart,5500,USD,2023,55536.0,2.5,GAS,AUTOMATIC,FWD,FRONT END,4.0,NaN,NaN
649,03/06/2026 18:00 UTC,Insurance,copart,10500,USD,2020,54505.0,3.5,GAS,AUTOMATIC,FWD,REAR END,4.0,MINOR DENT/SCRATCHES,4.0
997,03/06/2026 15:00 UTC,Insurance,copart,6000,USD,2025,NaN,2.5,HYBRID ENGINE,AUTOMATIC,AWD,REAR END,4.0,SIDE,4.0
1073,03/06/2026 20:00 UTC,Insurance,copart,5500,USD,2022,89059.0,2.5,GAS,AUTOMATIC,FWD,FRONT END,4.0,UNDERCARRIAGE,3.5


In [104]:
df_camry["fuel_type"].value_counts(ascending=False)

fuel_type
GAS              1729
HYBRID ENGINE     576
OTHER             343
Name: count, dtype: int64

In [105]:
df_camry["drive_type"].value_counts(ascending=False)

drive_type
FWD        2414
AWD         232
UNKNOWN       2
Name: count, dtype: int64

In [106]:
df_camry = df_camry.drop(columns=["drive_type"])

In [89]:
df_camry = df_camry[df_camry["fuel_type"] != "UNKNOWN"].copy()

In [90]:
df_camry.groupby("transmission").size().sort_values(ascending=False)

transmission
AUTOMATIC    2648
UNKNOWN        23
MANUAL          1
dtype: int64

In [91]:
df_camry = df_camry[df_camry["transmission"] == "AUTOMATIC"].copy()
df_camry = df_camry.drop(columns=["transmission"])

In [98]:
df_camry["primary_damage_severity"].value_counts(dropna=False)

primary_damage_severity
4.0    1192
NaN     650
3.5     461
3.0     152
4.5     107
2.5      46
2.0      25
1.5      13
1.0       2
Name: count, dtype: int64

In [99]:
df_camry["primary_damage"].value_counts(dropna=False)

primary_damage
FRONT END               1372
REAR END                 545
SIDE                     412
FRONT & REAR              95
ALL OVER                  46
ROLLOVER                  33
HAIL                      32
WATER/FLOOD               21
MINOR DENT/SCRATCHES      20
UNDERCARRIAGE             15
NORMAL WEAR & TEAR        12
MECHANICAL                10
VANDALISM                  6
BURN                       6
ROOF                       5
BIOHAZARD                  5
BURN - ENGINE              3
THEFT                      3
STORM DAMAGE               2
DAMAGE HISTORY             1
REPOSSESSION               1
FRAME DAMAGE               1
SUSPENSION                 1
ENGINE DAMAGE              1
Name: count, dtype: int64

In [100]:
low_count_damage = df_camry["primary_damage"].value_counts()
df_camry.loc[
    df_camry["primary_damage"].isin(low_count_damage[low_count_damage < 20].index),
    "primary_damage"
] = "OTHER"

In [92]:
df_camry["secondary_damage_severity"].value_counts(dropna=False)

secondary_damage_severity
NaN    1026
4.0     544
3.5     508
3.0     255
2.5     135
2.0      71
1.5      63
4.5      31
1.0      15
Name: count, dtype: int64

In [93]:
df_camry["secondary_damage"].value_counts(dropna=False)

secondary_damage
NaN                     876
FRONT END               600
SIDE                    440
REAR END                395
FRONT & REAR             95
MINOR DENT/SCRATCHES     62
UNDERCARRIAGE            50
HAIL                     29
MECHANICAL               20
ROOF                     14
ROLLOVER                 14
ALL OVER                 10
WATER/FLOOD               9
NORMAL WEAR & TEAR        6
BIOHAZARD                 5
BURN                      4
VANDALISM                 4
FRAME DAMAGE              3
THEFT                     3
STORM DAMAGE              2
STRIPPED                  2
REJECTED REPAIR           1
BURN - ENGINE             1
REPOSSESSION              1
SUSPENSION                1
ENGINE DAMAGE             1
Name: count, dtype: int64

In [94]:
pd.crosstab(
    df_camry["secondary_damage"].isna(),
    df_camry["secondary_damage_severity"].isna(),
    rownames=["Damage missing"],
    colnames=["Severity missing"]
)

Severity missing,False,True
Damage missing,,
False,1622,150
True,0,876


In [95]:
df_camry["secondary_damage"] = (df_camry["secondary_damage"].fillna("NOT_RECORDED"))
low_count_damage = df_camry["secondary_damage"].value_counts()
df_camry.loc[
    df_camry["secondary_damage"].isin(low_count_damage[low_count_damage < 20].index),
    "secondary_damage"
] = "OTHER"

In [96]:
df_camry["secondary_damage"].value_counts(dropna=False)

secondary_damage
NOT_RECORDED            876
FRONT END               600
SIDE                    440
REAR END                395
FRONT & REAR             95
OTHER                    81
MINOR DENT/SCRATCHES     62
UNDERCARRIAGE            50
HAIL                     29
MECHANICAL               20
Name: count, dtype: int64

In [109]:
df_camry.nunique()

sale_date                     383
seller_type                     2
auction                         2
sold_price                    456
year                            7
mileage                      2489
engine_volume                   2
fuel_type                       3
primary_damage                 10
primary_damage_severity         8
secondary_damage               10
secondary_damage_severity       8
dtype: int64

In [ ]:
# df_camry.to_csv("camry_cleaned.csv", index=False)

In [ ]:
df_camry = df_camry[(df_camry["mileage"] != 1) & (df_camry["mileage"] != 999999)].copy()

In [141]:
df_camry["sale_date"] = pd.to_datetime(
    df_camry["sale_date"],
    format="%m/%d/%Y %H:%M %Z",
    utc=True,
    errors="raise"
)

df_camry = df_camry.sort_values("sale_date").reset_index(drop=True)
df_camry.head(200)

,sale_date,seller_type,auction,sold_price,year,mileage,engine_volume,fuel_type,primary_damage,primary_damage_severity,secondary_damage,secondary_damage_severity
0,2025-12-23 14:30:00+00:00,Insurance,iaai,13250,2023,10964.0,2.5,GAS,FRONT END,4.5,FRONT END,4.0
1,2025-12-23 15:00:00+00:00,Seller,copart,8300,2021,114623.0,2.5,GAS,FRONT END,3.5,SIDE,2.5
2,2025-12-23 15:00:00+00:00,Insurance,copart,10900,2022,59171.0,2.5,GAS,FRONT END,4.0,SIDE,4.0
3,2025-12-23 15:00:00+00:00,Insurance,copart,8800,2024,30368.0,2.5,GAS,FRONT END,4.0,NOT_RECORDED,NaN
4,2025-12-23 15:00:00+00:00,Insurance,copart,17700,2025,6979.0,2.5,HYBRID ENGINE,FRONT END,4.0,NOT_RECORDED,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
195,2025-12-31 14:30:00+00:00,Insurance,iaai,11950,2024,24002.0,2.5,GAS,REAR END,3.5,REAR END,3.0
196,2025-12-31 14:30:00+00:00,Seller,iaai,16700,2025,21872.0,2.5,OTHER,FRONT END,4.0,FRONT END,4.0
197,2025-12-31 14:30:00+00:00,Seller,iaai,11300,2025,9411.0,2.5,OTHER,SIDE,4.0,SIDE,4.0
198,2025-12-31 14:30:00+00:00,Insurance,iaai,7650,2020,60087.0,2.5,GAS,SIDE,3.5,SIDE,2.5


In [143]:
X = df_camry.drop(columns=["sold_price", "sale_date"])
y = df_camry["sold_price"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=SEED)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)


In [144]:
numeric_features = [
    "year",
    "mileage",
    "engine_volume",
    "primary_damage_severity",
    "secondary_damage_severity"
]

categorical_features = [
    "seller_type",
    "auction",
    "fuel_type",
    "primary_damage",
    "secondary_damage"
]

In [151]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [153]:
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [155]:
linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['seller_type','auction','year',...,'primary_damage_severity', 'secondary_damage','secondary_damage_severity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be a

In [ ]:
models = {
    "Linear Regression": linear_pipeline,
    "Ridge": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(alpha=1.0))
    ]),
    "Lasso": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", Lasso(alpha=1.0, max_iter=10000))
    ])
}

results = []

for name, model in models.items():
    # Linear Regression was already fitted
    if name != "Linear Regression":
        model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_val, predictions),
        "RMSE": mean_squared_error(y_val, predictions) ** 0.5,
        "R²": r2_score(y_val, predictions)
    })

comparison = pd.DataFrame(results).sort_values("MAE")
comparison

,Model,MAE,RMSE,R²
2,Lasso,1937.239599,2471.544469,0.643222
0,Linear Regression,1937.851832,2471.984012,0.643095
1,Ridge,1938.003693,2472.284442,0.643009


In [175]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(max_iter=50000)
}

scores = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    result = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    scores[name] = {
        "MAE": -result["test_MAE"].mean(),
        "RMSE": -result["test_RMSE"].mean(),
        "R2": result["test_R2"].mean()
    }

pd.DataFrame(scores).T.round(3)

,MAE,RMSE,R2
Linear,1922.474,2547.735,0.602
Ridge,1921.564,2546.762,0.602
Lasso,1921.536,2547.239,0.602


In [177]:
alphas = np.logspace(-3, 4, 20)

results = {}

for name, model in {
    "Ridge": Ridge(),
    "Lasso": Lasso(max_iter=50000)
}.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    search = GridSearchCV(
        pipeline,
        param_grid={"model__alpha": alphas},
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    results[name] = search

    print(f"{name}")
    print("Best alpha:", search.best_params_["model__alpha"])
    print("Best CV MAE:", -search.best_score_)

Ridge
Best alpha: 2.069138081114788
Best CV MAE: 1925.3096236033987
Lasso
Best alpha: 4.832930238571752
Best CV MAE: 1920.8250182481702


In [178]:
validation_models = {
    "Linear Regression": linear_pipeline,
    "Ridge": results["Ridge"].best_estimator_,
    "Lasso": results["Lasso"].best_estimator_
}

validation_results = []

for name, model in validation_models.items():
    predictions = model.predict(X_val)

    validation_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_val, predictions),
        "RMSE": mean_squared_error(y_val, predictions) ** 0.5,
        "R²": r2_score(y_val, predictions)
    })

validation_comparison = (
    pd.DataFrame(validation_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

validation_comparison.round(3)

,Model,MAE,RMSE,R²
0,Linear Regression,1937.852,2471.984,0.643
1,Ridge,1938.356,2472.901,0.643
2,Lasso,1939.524,2474.018,0.643


In [180]:
best_name = validation_comparison.iloc[0]["Model"]
final_pipeline = clone(validation_models[best_name])

# Combine training and validation data
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

# Refit preprocessing and model together
final_pipeline.fit(X_train_val, y_train_val)

# Final evaluation on untouched test data
test_predictions = final_pipeline.predict(X_test)

print(f"Selected model: {best_name}")
print(f"Test MAE:  ${mean_absolute_error(y_test, test_predictions):,.2f}")
print(f"Test RMSE: ${mean_squared_error(y_test, test_predictions) ** 0.5:,.2f}")
print(f"Test R²:   {r2_score(y_test, test_predictions):.3f}")

Selected model: Linear Regression
Test MAE:  $1,961.09
Test RMSE: $2,567.77
Test R²:   0.585
